# Flow Match — Ligand Conformation Visualization

Overlay crystal pose (green) vs generated conformation (orange) in the pocket (gray).

**Prerequisites:** Run training first and save a checkpoint to `checkpoints/best_model.pt`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import Draw

from src.models.egnn import EGNNFlowModel, build_default_model
from src.models.flow_model import FlowMatcher
from src.data.dataset import PDBBindDataset, load_splits, make_dataloader
from src.training.metrics import kabsch_rmsd, strain_energy_ratio, mol_with_coords

print('Imports OK')

In [ ]:
# --- Config ---
CHECKPOINT  = '../checkpoints/best_model.pt'
SPLITS_PATH = '../data/splits.json'
PROCESSED   = '../data/processed'
RAW_DIR     = '../data/raw'
DEVICE      = torch.device('cpu')
N_DISPLAY   = 3   # number of test complexes to visualize

In [ ]:
# --- Load model ---
model = build_default_model()
ckpt  = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

flow_matcher = FlowMatcher(model, n_steps=20)

print(f"Loaded checkpoint from epoch {ckpt['epoch']}, val RMSD = {ckpt['val_rmsd']:.3f} Å")

In [ ]:
# --- Load test data ---
splits   = load_splits(SPLITS_PATH)
test_ids = splits['test'][:N_DISPLAY]
test_ds  = PDBBindDataset(PROCESSED, test_ids)
loader   = make_dataloader(test_ds, batch_size=1, shuffle=False)

print(f'Visualizing {len(test_ids)} complexes: {test_ids}')

In [ ]:
# --- Generate conformations and compute RMSD ---
records = []

for batch in loader:
    cid      = batch.complex_id[0] if isinstance(batch.complex_id, list) else batch.complex_id
    crystal  = batch['ligand'].pos  # [N, 3]
    
    generated = flow_matcher.generate(batch)[0]  # [N, 3]
    rmsd = kabsch_rmsd(generated, crystal.cpu())
    
    # Load RDKit mol from raw SDF
    sdf_path = os.path.join(RAW_DIR, cid, f'{cid}_ligand.sdf')
    mol = Chem.MolFromMolFile(sdf_path, removeHs=True) if os.path.exists(sdf_path) else None
    
    records.append({
        'cid': cid,
        'crystal': crystal.cpu().numpy(),
        'generated': generated.cpu().numpy(),
        'rmsd': rmsd,
        'mol': mol,
    })
    print(f'  {cid}: RMSD = {rmsd:.3f} Å')

In [ ]:
# --- Helper: coords → SDF string ---
def mol_to_sdf_str(mol, coords):
    """Assign coords to mol and return SDF block string."""
    if mol is None:
        return None
    mol_new = mol_with_coords(mol, coords)
    return Chem.MolToMolBlock(mol_new)


def load_pocket_pdb_str(cid):
    pdb_path = os.path.join(RAW_DIR, cid, f'{cid}_protein.pdb')
    if not os.path.exists(pdb_path):
        return None
    with open(pdb_path) as f:
        return f.read()

In [ ]:
# --- Visualize each complex ---
for rec in records:
    cid = rec['cid']
    print(f'\n=== {cid}  (RMSD = {rec["rmsd"]:.3f} Å) ===')
    
    crystal_sdf   = mol_to_sdf_str(rec['mol'], rec['crystal'])
    generated_sdf = mol_to_sdf_str(rec['mol'], rec['generated'])
    pocket_pdb    = load_pocket_pdb_str(cid)
    
    view = py3Dmol.view(width=700, height=450)
    
    model_idx = 0
    
    # Crystal pose: green sticks
    if crystal_sdf:
        view.addModel(crystal_sdf, 'sdf')
        view.setStyle({'model': model_idx}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15}})
        model_idx += 1
    
    # Generated pose: orange sticks
    if generated_sdf:
        view.addModel(generated_sdf, 'sdf')
        view.setStyle({'model': model_idx}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.15}})
        model_idx += 1
    
    # Pocket: gray cartoon
    if pocket_pdb:
        view.addModel(pocket_pdb, 'pdb')
        view.setStyle({'model': model_idx}, {
            'cartoon': {'color': 'lightgray', 'opacity': 0.4},
            'stick': {'colorscheme': 'grayCarbon', 'radius': 0.05, 'opacity': 0.3},
        })
    
    view.zoomTo()
    view.show()
    
    print('  Green = crystal pose | Orange = generated')

In [ ]:
# --- RMSD table ---
print(f'{'Complex':<12} {'RMSD (Å)':>10}')
print('-' * 24)
for rec in records:
    print(f"{rec['cid']:<12} {rec['rmsd']:>10.3f}")

all_rmsds = [r['rmsd'] for r in records]
print('-' * 24)
print(f"{'Median':<12} {np.median(all_rmsds):>10.3f}")
print(f"{'Mean':<12} {np.mean(all_rmsds):>10.3f}")

In [ ]:
# --- Full test set RMSD histogram ---
# Run this cell after full evaluation (compute_test_metrics) to compare distributions.

full_test_ds = PDBBindDataset(PROCESSED, splits['test'])
full_loader  = make_dataloader(full_test_ds, batch_size=4, shuffle=False)

rmsd_model  = []
rmsd_etkdg  = []  # populated if ETKDG baseline is computed separately

with torch.no_grad():
    for batch in full_loader:
        lig_batch   = batch['ligand'].batch
        crystal_pos = batch['ligand'].pos
        generated   = flow_matcher.generate(batch)
        n_graphs    = lig_batch.max().item() + 1
        for g in range(n_graphs):
            mask = lig_batch == g
            rmsd_model.append(kabsch_rmsd(generated[g].cpu(), crystal_pos[mask].cpu()))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rmsd_model, bins=20, alpha=0.75, color='steelblue', label='Flow Match')
ax.axvline(np.median(rmsd_model), color='navy',  linestyle='--', label=f'Median {np.median(rmsd_model):.2f} Å')
ax.axvline(3.0, color='red', linestyle=':', alpha=0.7, label='Target (3.0 Å)')
ax.set_xlabel('Kabsch RMSD (Å)')
ax.set_ylabel('Count')
ax.set_title('Test Set RMSD Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('../results_rmsd_histogram.png', dpi=150)
plt.show()

print(f'Test median RMSD: {np.median(rmsd_model):.3f} Å  (target < 3.0 Å)')